### Outstanding Questions
* Of all apps submitted during {time frame}, how many days have they been/did they spend waiting to be approved?

In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

In [ ]:
import datetime
from pypika import MySQLQuery, Table, Order, functions as fn

def create_query_devs_submitted_since(date="2014-01-01",
                                      approval_statuses=["NEW", "PENDING", "APPROVED", "DENIED"]):
    developer = Table("developer")
    account = Table("account")

    q = MySQLQuery.from_(developer).join(
        account
    ).on(
        developer.owner_account_id == account.id
    ).select(
        account.email,
        developer.uuid,
        developer.name,
        developer.first_submitted_time,
        developer.approval_status
    ).where(
        account.email.not_like('%clover.com')
    ).where(
        developer.name != "Clover"
    ).where(
        (developer.approval_status.isin(approval_statuses)) &
        (developer.first_submitted_time.notnull())
    ).where(
        developer.first_submitted_time >= fn.Date(date)
    ).orderby(developer.first_submitted_time, order=Order.asc)

    print(q)
    query = q.get_sql()
    return query
    
# Year to date
start_date = datetime.datetime.utcnow().replace(month=1, day=1).date()

pending_devs_query = create_query_devs_submitted_since(approval_statuses=["PENDING"])
db = Db("~/.clover/p801.cfg")
df = pd.read_sql(pending_devs_query, con=db.conn)
db.close()

df

In [ ]:
import datetime

def days_since(x):
    current = datetime.datetime.utcnow()
    diff = current - x
    return diff.days
    
df["days_pending"] = df["first_submitted_time"].map(lambda x: days_since(x))
df.head()

In [ ]:
# Plot length pending as a histogram.

plt.hist(df['days_pending'].dropna(), color='green', edgecolor='black', bins=int(df['days_pending'].max()/30))
plt.title('Histogram of Pending Apps')
plt.xlabel('Days Pending')
plt.ylabel('Apps')

print(df["days_pending"].describe())

# print "Range of days pending: " + str(df['days_pending'].min()) + ", " + str(df['days_pending'].max())
# print "Mean (average) days pending: " + str(df['days_pending'].mean())
# print "Median days pending: " + str(df['days_pending'].median())
# print "Modal days pending: " + str(df['days_pending'].mode().values[0])

In [ ]:
submitted_devs_query = create_query_devs_submitted_since(date=start_date)
db = Db("~/.clover/p801.cfg")
df = pd.read_sql(submitted_devs_query, con=db.conn)
db.close()

df.head()

In [ ]:
print(df['approval_status'].describe())
print(df['approval_status'].value_counts())
df['approval_status'].value_counts(normalize=True)

In [ ]:
CountStatus = pd.value_counts(df["approval_status"].values, sort=True)
CountStatus.plot.barh(title="Status of Devs Submitted Since " + start_date.strftime("%b %-d %Y"))